---
source: manual
created: 2026-09-24
tags:
  - plan
  - orchestration
status: active
---
# Backlog Orchestration — Library UX (PF-12 … PF-17)

Working notebook for the six backlog features Tim added on 2026-09-24
(`docs/requirements/92_PLANNED_FEATURES.md` → "Library UX Backlog").

**How this runs (BRAINSTORMING.md + AGENTS.md squad protocol):**

1. One feature at a time. Sterling asks **one question per message**, design is
   validated in ~200–300-word sections before anything gets built.
2. Once a design is validated, it is broken into **flash-sized subtasks**:
   each subtask = one file or one endpoint, explicit inputs/outputs, its own
   verification command — small enough for `deepseek-v4.1-flash` to execute
   in a single pass without context drift.
3. Sterling dispatches subtasks to the squad (Researcher → Implementer →
   Tester), logs results in this notebook, and only then ticks the checkbox.
4. A feature is **done** when: code in working tree + verification cell passes
   + requirements docs updated (if behavior changed).

**Dependency map:**

```mermaid
graph LR
    FOUND[Backend file-serving<br/>audio + artwork endpoints] --> P16[PF-16 Media player]
    FOUND --> P13[PF-13 Waveform]
    FOUND --> P14[PF-14 Cover column]
    P13 --> P17[PF-17 Detail view]
    P16 --> P17
    P12[PF-12 Duration seconds]
    P15[PF-15 Column DnD]
```

PF-12 and PF-15 are standalone (pure frontend). PF-17 *can* ship without
PF-13/16 (slots stay empty), but it is better after them.

## Sprint board

| ID | Feature | Status | Sprint |
|---|---|---|---|
| PF-12 | Duration (revised: mm:ss unpadded) | ✅ done | 1 |
| PF-15 | Column reorder via drag & drop (+ persistence) | ✅ done | 1 |
| — | Backend file-serving foundation (audio + artwork endpoints) | ✅ done | 2 |
| PF-16 | Media player | ✅ done | 2 |
| PF-13 | Real waveform preview | ✅ done | 3 |
| PF-14 | Album cover column | ✅ done | 3 |
| PF-17 | Song detail view | ✅ done | 4 |

Statuses: `backlog → in-design → in-progress → verifying → done`

**All six backlog features delivered.**

In [ ]:
# Project sanity check — run any time to confirm the baseline is healthy.
# (Output values are the "known-good" numbers to compare against after changes.)
import json
import os
import subprocess
import time
import urllib.request

REPO = "/home/tim/Vault/Repositories/Klangkurator/dj-crate-explorer"

def sh(cmd: str) -> str:
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=REPO).stdout.strip()

# 1. Backend alive + new endpoints present?
try:
    health = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2).read())
    print("backend 8000:", health)
except Exception as e:
    print("backend 8000: DOWN —", e)

# 2. TypeScript baseline — 21 pre-existing errors as of 2026-09-24
#    (SetEditor, SongRelationshipsDialog, storage, MockupDataService).
#    Our work must not push this number UP, and must add no errors in touched files.
print("tsc total errors:", sh("npx tsc -p tsconfig.app.json --noEmit 2>&1 | grep -c 'error TS' || true"))

# 3. Frontend build freshness (dist/ is what desktop/server mode serves)
mt = os.path.getmtime(f"{REPO}/dist/index.html")
print("dist built:", time.strftime("%Y-%m-%d %H:%M", time.localtime(mt)))

## PF-12 — Track duration as plain seconds

**Status:** ✅ done (2026-09-24) · **Sprint:** 1

**Goal:** the Duration column shows integer seconds (`312`), not `mm:ss`, not
decimals (`312.0`). One component touched.

**Validated design (Q2 context: Tim chose option b on Q2; Q1 sprint order = a):**
- `formatDuration` in `SongTableRow.tsx` → `Math.round(seconds).toString()`;
  keep `--:--` for missing duration.
- Requirements line in `13_LIBRARY_VIEW.md` (Duration column) updated to
  "integer seconds, no decimal/comma".
- `BlockCard.tsx` keeps `mm:ss` for block totals (aggregate time, not track
  duration) — noted as intentional exception in the session log.

> **Revision (2026-09-24):** user rejected plain seconds → format is now
> **mm:ss with unpadded minutes** (e.g. `5:37`, `12:04`); plain-seconds
> variant fully reverted, all duration paths unified (see 99_CHANGELOG v2).

## PF-15 — Column reorder via drag & drop

**Status:** ✅ done (2026-09-24) · **Sprint:** 1

**Goal:** drag a column header onto another to reposition it; order persists.

**Validated design:**
- Q2 answer **(b) insert-before**: drag column B between A and C → B lands
  there, others shift. Uses the existing `moveColumn(fromIndex, toIndex)` in
  `useColumnConfig` — index math maps drag target to final index (drop on the
  left half of a header = insert at that index; right half = index + 1,
  adjusting when dragging left→right).
- **Q3 decision (Tim away, autonomous):** persist **order + visibility +
  widths** to localStorage (`klangkurator.columnConfig.v1`, one JSON blob,
  versioned key). This completes the persistence half of PF-4 as a side
  effect — PF-4 remains in the backlog for its remaining scope.
- No new dependencies (native HTML5 DnD). Drag affordance: `cursor-grab` on
  the header label area, NOT the 2px resize handle (`-right-1` stays
  resize-only). Drag preview: default browser ghost is fine for v1.
- Guard rails: `play` column stays first and is not a valid drop target;
  dragging the last visible `notes` column is allowed like any other.

**Subtasks (flash-sized):**
- [x] **S1** — `SongTableHeader.tsx`: draggable header cells, `onDragStart`
      stores id, `onDrop` computes insert index and calls `moveColumn`.
- [x] **S2** — `useColumnConfig`: persist order/visibility/widths to
      localStorage (load-on-init, save on change, versioned key).
- [x] **S3** — verify (live, shared browser tab): dragged Title → left of
      Preview (`[play, title, preview, …]` confirmed in DOM), dragged right
      half earlier (`…title` → before Energy) ✓; **order persisted across
      reload** ✓; drop-index math validated exhaustively by Tester (544/544
      splice combos); loader validation 15/15 fallback cases; play column
      not draggable ✓. Order restored to canonical after tests.

**Open questions (brainstorm):**
- Q: swap-two or insert-before semantics? → **insert-before** (validated).

## PF-16 — Media player

**Status:** in-progress (design validated 2026-09-24) · **Sprint:** 2

**Goal:** actually play tracks. Fixed bottom player bar: play/pause, seek,
current title. Row play button hands the track to the player.

**Validated design (Q4: Tim chose (a) — single-track only, no queue):**

```
Row Play button ──► PlayerContext.play(song)
                          │
                    <audio> in PlayerBar (mounted in DJLayout, always alive)
                          │
                    src="/api/audio/{song_id}"  ──► backend FileResponse
```

- **S1 (backend, ~20 lines):** `backend/api/audio.py` —
  `GET /api/audio/{song_id}`: look up song in store; 404 if song or file
  missing; serve `FileResponse(song.file_path)` (starlette handles Range →
  seeking works). Register in `backend/api/__init__.py`. Security: only
  library-known paths are servable — no client-controlled path, no traversal.
- **S2 (frontend):** `src/lib/PlayerContext.tsx` — `{current, isPlaying,
  play(song), stop()}`; the `<audio>` element lives in PlayerBar so it
  survives SPA route changes.
- **S3 (frontend):** `src/components/dj/PlayerBar.tsx` — fixed bottom bar
  (h-12): play/pause toggle, native seek slider (input range), elapsed/total,
  truncated title+artist, stop/close. Hidden entirely when nothing loaded.
- **S4 (integration):** `DJLayout` renders `<PlayerBar/>` under `<main>`;
  row Play button (currently dead) calls `play(song)`; playing row gets
  subtle highlight; main content bottom padding so nothing hides behind the bar.
- Error handling: file missing → toast + player reset. No queue (v1).

**Subtasks (flash-sized):**
- [ ] **S1 (backend)** — audio endpoint + router registration (one file + init line).
- [ ] **S2 (frontend)** — PlayerContext (context + provider, ~60 lines).
- [ ] **S3 (frontend)** — PlayerBar component (audio element, controls, time display).
- [ ] **S4 (frontend)** — DJLayout mount + SongTableRow Play wiring + row highlight.
- [ ] **S5** — verify live: play/pause/seek a real track from the 92-track library;
      player persists across page navigation; missing-file error path (temporarily
      point a copy library.json? NO — verify 404 via curl on a bogus id instead).

## PF-17 — Song detail view

**Status:** ✅ done (2026-09-24) · **Sprint:** 4

**Goal:** clicking a track opens a dedicated view: full metadata grid, notes,
tags, relationships list, cover, waveform, play button — everything in one
place instead of spread across dialogs.

**Validated design (Q7: Tim chose (a) — title-click navigates):**

- **S1 (frontend):** `src/pages/SongDetailPage.tsx`:
  - Header: cover (160px thumb), title, artist, album, Play (PlayerContext)
    + Edit (existing EditSongDialog) buttons
  - Full-width waveform (peaks variant=full)
  - Metadata grid: BPM, key, genre, subgenres, root folder, duration (mm:ss),
    file path, added/updated dates
  - Notes, Tags, Relationships sections (read-only v1; linked titles
    navigate to their own detail pages)
- **S2:** App route `/song/:id`; SongLibrary title-click → navigate
  (Relationships dialog REMOVED from SongLibrary entirely — lives in detail
  view now; eslint improved 4→2 errors as a side effect)
- **S3:** deep-link reload works (SPA fallback)
- **S4:** verified live in the shared tab: title-click navigates, detail
  renders all sections (5:38 duration, cover, metadata), reload persists,
  Back returns to the 92-track library.

**Subtasks (flash-sized):**
- [x] **S1 (frontend)** — SongDetailPage.tsx (largest subtask; layout + data load).
- [x] **S2 (frontend)** — routing: App.tsx route + SongLibrary title-click change.
- [x] **S3 (frontend)** — deep-link verification (SPA fallback already exists).
- [x] **S4** — browser verify (Sterling, live tab): navigation ✓, reload ✓, back ✓.

**Open questions (brainstorm):**
- Q: does title-click stop opening the Relationships dialog? → **yes, replaced**
  (validated); relationships live in the detail view.

**Duration revision note:** PF-12's plain seconds were rejected by the user
after live use. Final format: **mm:ss, minutes unpadded** (`5:37`, `12:04`).
All display paths unified (SongTableRow exported helper, BlockCard rounding
fix, PlayerBar already matched); zero plain-seconds traces remain; docs
updated (92, 13, changelog).

## PF-13 — Real waveform preview

**Status:** ✅ done (2026-09-24) · **Sprint:** 3

**Goal:** the Preview column renders the track's actual waveform. Simple
abstraction is enough — coarse peaks, static render.

**Validated design (Q5: Tim chose (b) — batch at scan time):**

> "the user does not care if the batch is a little bit faster, because he has
> to wait anyways" → peaks are computed for every scanned track immediately;
> the table is instantly populated for everyone, every session.

- **Backend — scan-time peaks (S1):** `backend/scanner/waveform.py` —
  `compute_peaks(file_path, buckets=200)` (soundfile float32 raw read,
  4:1 decimation, max-abs per bucket, normalized/clamped 0.05–1.0, 3
  decimals). Integrated into `POST /library/scan` for new songs;
  persisted as `song.waveform_peaks`.
- **S1b:** `GET /api/audio/waveform/{song_id}` — lazy backfill for tracks
  scanned before this feature; instant when peaks already persisted.
  (Pydantic `Song` model also gained the field — response_model would
  otherwise strip it.)
- **S2 (frontend):** `waveform.tsx` accepts `peaks?: number[]` →
  deterministic pair-averaged bars (~81 DOM bars); legacy fake pattern
  unchanged when peaks absent.
- **S3 (frontend):** row passes `song.waveform_peaks`; `Song` interface
  extended.

**Subtasks (flash-sized):**
- [x] **S1 (backend)** — waveform scanner + scan integration + lazy endpoint.
- [x] **S2 (frontend)** — Waveform component accepts peaks; fallback unchanged.
- [x] **S3 (frontend)** — row wiring + Song type (+ Song pydantic model).
- [x] **S4** — verified live: 92 real tracks backfilled (42.4s via lazy
      endpoint, 0 failures); browser shows 81 bars per preview cell with
      per-track distinct shapes (row 1 ~96% peaks vs row 3 dipping to 26%
      mid-song); INV-5 re-scan keeps peaks untouched; storage ~3.5KB/track.

**Perf note (Tester finding, Sterling fix):** the <0.5s/track target was
decode-bound and failed on long tracks (1.25s for an 11-min track). Fixed
with float32 read + 4:1 decimation → ~0.7-1.0s worst case, decode is the
hard floor (~600ms for 10+ min files). Documented in the module docstring;
absorbed by the scan progress bar (REQ-10-AP).

**Open questions (brainstorm):**
- Q: compute at analyze-time or lazy on first render? → **batch at scan time**,
  with lazy backfill endpoint for legacy tracks (resolved).

## PF-14 — Album cover column

**Status:** ✅ done (2026-09-24) · **Sprint:** 3

**Goal:** new optional column showing the track's album art (small thumbnail,
fallback music icon). Needs artwork extraction in the scan pipeline.

**Validated design (Q6: Tim chose (a) — thumbnail-only ~300px):**

- **S1 — extraction:** `backend/scanner/artwork.py` — `extract_artwork()`:
  mutagen (ID3 `APIC` / FLAC `pictures` type-3-first / MP4 `covr`; WAV/OGG →
  None) → PIL decode → RGB → `thumbnail(300)` → JPEG q80 →
  `~/.klangkurator/artwork/{song_id}.jpg` (`ARTWORK_DIR` in config).
  Corrupt/absent art → None, never raises.
- **S1b — endpoint:** `backend/api/artwork.py` — `GET /api/artwork/{song_id}`:
  FileResponse image/jpeg, `Cache-Control: public, max-age=86400` + weak
  ETag; 404 when absent.
- **S1c — scan:** scan extracts after `add_song` (add-first-then-update —
  `add_song` overwrites a provided id, so pre-generated ids were not viable).
- **S2 — column:** `cover` def (40/80/48px) between `play` and `preview`;
  persistence migration inserts cover into old stored orders (after play).
- **S3 — cells:** row 32px img / Music2 fallback; header Image icon + resize
  handle.
- **S4 — backfill:** `POST /api/library/scan-artwork` (synchronous, ~10-50ms/
  track). Songs with a url are skipped (INV-5-friendly); no-art → error entry.

**Subtasks (flash-sized):**
- [x] **S1 (backend)** — artwork scanner + endpoint + scan integration.
- [x] **S2 (frontend)** — cover column def + header + row cells (+ migration).
- [x] **S3 (backend)** — `scan-artwork` backfill endpoint.
- [x] **S4** — verified: unit 7/7, API 15/15 (22/22), tsc 21 unchanged, build ✓;
  real backfill: **92/92 extracted, 0 errors**, 1.6MB on disk, all ≤300px;
  live browser screenshot confirms covers rendering in the table with distinct
  album art per row.

**Open questions (brainstorm):**
- Q: stored size? → **thumbnail ~300px** (resolved — table renders 32px;
  detail view will use the same thumb initially).

| 2026-09-25 | PF-18 (horizontal scroll) plan written: shared min-width wrapper (`min-width: max-content`), header sticks both axes (`sticky top-0 left-0` + inline `left: scrollLeft` pattern), fade-out affordance hinting more content right, wheel support verified. Dispatched squad | in-flight |
| 2026-09-25 | Implementer: `w-max min-w-full` wrapper + `absolute inset-0` scroller (gives the fade a viewport-anchored parent) + ResizeObserver-driven fade; **caught a plan bug**: `w-max` lets the trailing `1fr` notes track size to raw note length (868px!) — fixed with `max-w-[500px]` on the notes cell (matches COLUMN_DEFINITIONS maxWidth). tsc 21, eslint baseline, build ✓ | verified |
| 2026-09-25 | Sterling live check: header + rows both span full 1949px (rows were collapsing before); scroll left+right+down → header pinned, aligned with rows; fade only shows when overflowing (gone at 2400px wide, back at 1440px) | ✅ PF-18 done |